In [1]:
#!/usr/bin/env python
# coding: utf-8

import os
import hues
import pickle
import numpy as np
import pandas as pd

from tools import CustomScaler, check_folder

In [2]:
# =========================
# 0) 可调参数（先按推荐默认值）
# =========================
WINDOW_SIZE = 48  # 输入历史窗口长度
HORIZON_SIZE = 12  # 预测窗口长度

# 训练/验证划分（注意：WADI 的测试集由 Train==0 单独给出，因此这里会把 Train==1 的数据再切 train/val）
TRAIN_RATIO = 0.7

# 是否下采样（None 表示不下采样；10 表示 10 秒采样）
DOWNSAMPLE_SEC = 10  # 推荐 10；如要保持 1s：改成 None

# 保存路径（你可按自己工程目录改）
save_path = f"E:\DataForCode/5_AnomalyDetection/WADI_STGNN_[{WINDOW_SIZE}_{HORIZON_SIZE}_{DOWNSAMPLE_SEC}]/"

In [3]:
df = pd.read_pickle('../../Data/WADI/data.pkl')

hues.info(df.shape)

df.head()

18:58:13 - INFO - (1382401, 101)


,Timestamp,Train,1_AIT_001_PV,1_AIT_002_PV,1_AIT_003_PV,1_AIT_004_PV,1_AIT_005_PV,1_FIT_001_PV,1_LT_001_PV,1_MV_001_STATUS,...,3_AIT_001_PV,3_AIT_002_PV,3_AIT_003_PV,3_AIT_004_PV,3_AIT_005_PV,3_FIT_001_PV,3_LT_001_PV,LEAK_DIFF_PRESSURE,TOTAL_CONS_REQUIRED_FLOW,Attack
0,2017-09-25 18:00:00,1,171.155,0.619473,11.5759,504.645,0.318319,0.001157,47.8911,1,...,0.0,0.0,1.18417,618.079,0.375448,0.001268,64.3425,67.9651,0.68,0
1,2017-09-25 18:00:01,1,171.155,0.619473,11.5759,504.645,0.318319,0.001157,47.8911,1,...,0.0,0.0,1.18417,618.079,0.375448,0.001268,64.3425,67.9651,0.68,0
2,2017-09-25 18:00:02,1,171.155,0.619473,11.5759,504.645,0.318319,0.001157,47.8911,1,...,0.0,0.0,1.18417,618.079,0.375448,0.001268,64.3425,67.9651,0.68,0
3,2017-09-25 18:00:03,1,171.155,0.607477,11.5725,504.673,0.318438,0.001207,47.7503,1,...,0.0,0.0,1.18005,618.107,0.375398,0.001133,64.3199,67.1948,0.68,0
4,2017-09-25 18:00:04,1,171.155,0.607477,11.5725,504.673,0.318438,0.001207,47.7503,1,...,0.0,0.0,1.18005,618.107,0.375398,0.001133,64.3199,67.1948,0.68,0


In [4]:
# 缺失值检查
df.isna().sum().sum()

0

In [5]:
# =========================
# 3) 列类型划分：特征列 / 标签列
# =========================
label_cols = ["Timestamp", "Train", "Attack"]

# 特征列：除 Timestamp/Train/Attack 外的所有列
feature_cols = [c for c in df.columns if c not in label_cols]

# 状态列（离散/开关/阀门等）：WADI 通常以 STATUS 命名
status_cols = [c for c in feature_cols if "STATUS" in c.upper()]

# 连续列：其余都按连续量处理（PV、流量、压力等）
cont_cols = [c for c in feature_cols if c not in status_cols]

hues.info(f"Feature cols [{len(feature_cols)}]: ", str(feature_cols))
hues.info(f"Status cols [{len(status_cols)}]: ", str(status_cols))
hues.info(f"Continuous cols [{len(cont_cols)}]: ", str(cont_cols))

18:58:13 - INFO - Feature cols [98]:  ['1_AIT_001_PV', '1_AIT_002_PV', '1_AIT_003_PV', '1_AIT_004_PV', '1_AIT_005_PV', '1_FIT_001_PV', '1_LT_001_PV', '1_MV_001_STATUS', '1_MV_002_STATUS', '1_MV_003_STATUS', '1_MV_004_STATUS', '1_P_001_STATUS', '1_P_003_STATUS', '1_P_005_STATUS', '1_P_006_STATUS', '2_DPIT_001_PV', '2_FIC_101_CO', '2_FIC_101_PV', '2_FIC_101_SP', '2_FIC_201_CO', '2_FIC_201_PV', '2_FIC_201_SP', '2_FIC_301_CO', '2_FIC_301_PV', '2_FIC_301_SP', '2_FIC_401_CO', '2_FIC_401_PV', '2_FIC_401_SP', '2_FIC_501_CO', '2_FIC_501_PV', '2_FIC_501_SP', '2_FIC_601_CO', '2_FIC_601_PV', '2_FIC_601_SP', '2_FIT_001_PV', '2_FIT_002_PV', '2_FIT_003_PV', '2_FQ_101_PV', '2_FQ_201_PV', '2_FQ_301_PV', '2_FQ_401_PV', '2_FQ_501_PV', '2_FQ_601_PV', '2_LS_101_AH', '2_LS_101_AL', '2_LS_201_AH', '2_LS_201_AL', '2_LS_301_AH', '2_LS_301_AL', '2_LS_401_AH', '2_LS_401_AL', '2_LS_501_AH', '2_LS_501_AL', '2_LS_601_AH', '2_LS_601_AL', '2_LT_001_PV', '2_LT_002_PV', '2_MCV_007_CO', '2_MCV_101_CO', '2_MCV_201_CO', '

In [6]:
# 检查状态开关列的数据分布情况.
# 统计每个 STATUS 列中不同取值的样本数量
for c in status_cols:
    vc = df[c].value_counts(dropna=False).sort_index()
    hues.log(vc)

18:58:13 - 1_MV_001_STATUS
0       2878
1    1002387
2     377136
Name: count, dtype: int64
18:58:13 - 1_MV_002_STATUS
0         50
1    1380864
2       1487
Name: count, dtype: int64
18:58:13 - 1_MV_003_STATUS
0         55
1    1380864
2       1482
Name: count, dtype: int64
18:58:13 - 1_MV_004_STATUS
0      2959
1    995287
2    384155
Name: count, dtype: int64
18:58:13 - 1_P_001_STATUS
1    1002537
2     379864
Name: count, dtype: int64
18:58:13 - 1_P_003_STATUS
1    1002575
2     379826
Name: count, dtype: int64
18:58:13 - 1_P_005_STATUS
1    1068037
2     314364
Name: count, dtype: int64
18:58:13 - 1_P_006_STATUS
1    1382317
2         84
Name: count, dtype: int64
18:58:13 - 2_MV_003_STATUS
0      14137
1    1053861
2     314403
Name: count, dtype: int64
18:58:13 - 2_MV_006_STATUS
0      13913
1    1142543
2     225945
Name: count, dtype: int64
18:58:13 - 2_MV_101_STATUS
0      1056
1    729078
2    652267
Name: count, dtype: int64
18:58:13 - 2_MV_201_STATUS
0       993
1    675734

In [7]:
# =========================
# 4) 切分 Train==1 / Train==0
# =========================
df_train_val = df[df["Train"] == 1].copy()
df_test = df[df["Train"] == 0].copy()

hues.info(("TrainVal raw:", df_train_val.shape))
hues.info(("Test raw:", df_test.shape))

# 只保留必要列（仍保留 Timestamp/Train/Attack，便于下采样聚合）
df_train_val = df_train_val[["Timestamp", "Train", "Attack"] + feature_cols]
df_test = df_test[["Timestamp", "Train", "Attack"] + feature_cols]

18:58:14 - INFO - ('TrainVal raw:', (1209600, 101))
18:58:14 - INFO - ('Test raw:', (172801, 101))


In [8]:
from tools import downsample_block

if DOWNSAMPLE_SEC is not None:
    df_train_val = downsample_block(df_train_val, cont_cols, status_cols, DOWNSAMPLE_SEC)
    df_test = downsample_block(df_test, cont_cols, status_cols, DOWNSAMPLE_SEC)

hues.info(("TrainVal after downsample:", df_train_val.shape))
hues.info(("Test after downsample:", df_test.shape))

18:58:16 - INFO - ('TrainVal after downsample:', (120960, 101))
18:58:16 - INFO - ('Test after downsample:', (17281, 101))


In [9]:
# =========================
# 6) 训练/验证按时间顺序切分（在 Train==1 内部再切）
#    说明：
#    - 由于测试集由 Train==0 单独给出，我们不再保留“train/val/test=0.7/0.2/0.1”的第三段
#    - 这里把 Train==1 的数据全部用完：train_ratio = 0.7/(0.7+0.2), val_ratio = 0.2/(0.7+0.2)
# =========================
tv_len = len(df_train_val)

split_idx = int(tv_len * TRAIN_RATIO)

df_train = df_train_val.iloc[:split_idx].copy()
df_val = df_train_val.iloc[split_idx:].copy()

hues.info("df_train:", df_train.shape)
hues.info("df_val:", df_val.shape)
hues.info("df_test:", df_test.shape)

18:58:16 - INFO - df_train: (84672, 101)
18:58:16 - INFO - df_val: (36288, 101)
18:58:16 - INFO - df_test: (17281, 101)


In [10]:
# =========================
# 7) 构造 scaler（只用 train 拟合，避免数据泄露）
#    连续列：z-score（按列）
#    状态列：不归一化（把 mean=0,std=1，相当于 transform 后原值不变）
# =========================
train_values = df_train[feature_cols].values.astype(np.float64)

scaler = CustomScaler(train_values, use_one=False)

# 让状态列保持原值：mean=0,std=1
if len(status_cols) > 0:
    status_idx = [feature_cols.index(c) for c in status_cols]
    scaler.mean[status_idx] = 0.0
    scaler.std[status_idx] = 1.0

hues.success("Scaler fitted on df_train. Status cols are kept as-is (mean=0,std=1).")

请注意以下列的标准差为零:
列 [8] 标准差为零, 所有值相同.
列 [9] 标准差为零, 所有值相同.
列 [14] 标准差为零, 所有值相同.
列 [57] 标准差为零, 所有值相同.
列 [77] 标准差为零, 所有值相同.
18:58:16 - SUCCESS - Scaler fitted on df_train. Status cols are kept as-is (mean=0,std=1).


In [11]:
# =========================
# 8) 标准化并生成滑动窗口样本 (x,y)
#    x: [num_samples, WINDOW_SIZE, num_nodes, 1]
#    y: [num_samples, HORIZON_SIZE, num_nodes, 1]
# =========================
def build_xy_from_df(_df: pd.DataFrame) -> tuple[np.ndarray, np.ndarray]:
    # 只取特征列
    data_raw = _df[feature_cols].values.astype(np.float64)

    # 标准化
    data_std = scaler.transform(data_raw).astype(np.float32)

    # 增加最后一维 feature=1
    data_std = np.expand_dims(data_std, axis=-1)  # [T, N, 1]

    T, N, F = data_std.shape

    # 可生成样本数：从 t=WINDOW-1 开始，到 t=T-HORIZON-1 结束（含）
    num_samples = T - (WINDOW_SIZE + HORIZON_SIZE) + 1
    if num_samples <= 0:
        raise ValueError(f"Not enough timesteps: T={T}, WINDOW={WINDOW_SIZE}, HORIZON={HORIZON_SIZE}")

    x = np.zeros((num_samples, WINDOW_SIZE, N, 1), dtype=np.float32)
    y = np.zeros((num_samples, HORIZON_SIZE, N, 1), dtype=np.float32)

    # 逐样本填充（尽量保持逻辑直观，后续也好改）
    for i in range(num_samples):
        t0 = i
        x[i] = data_std[t0: t0 + WINDOW_SIZE]
        y[i] = data_std[t0 + WINDOW_SIZE: t0 + WINDOW_SIZE + HORIZON_SIZE]

    return x, y


x_train, y_train = build_xy_from_df(df_train)
x_val, y_val = build_xy_from_df(df_val)
x_test, y_test = build_xy_from_df(df_test)

hues.info("train x:", x_train.shape, "y:", y_train.shape)
hues.info("val   x:", x_val.shape, "y:", y_val.shape)
hues.info("test  x:", x_test.shape, "y:", y_test.shape)

18:58:17 - INFO - train x: (84613, 48, 98, 1) y: (84613, 12, 98, 1)
18:58:17 - INFO - val   x: (36229, 48, 98, 1) y: (36229, 12, 98, 1)
18:58:17 - INFO - test  x: (17222, 48, 98, 1) y: (17222, 12, 98, 1)


In [12]:
check_folder(save_path)

# 保存数据
for cat in ["train", "val", "test"]:
    _x = locals()[f"x_{cat}"]
    _y = locals()[f"y_{cat}"]

    np.savez_compressed(
        os.path.join(save_path, f"{cat}.npz"),
        x=_x.astype(np.float32),
        y=_y.astype(np.float32),
    )
    hues.success(f"Saved {cat}.npz -> x:{_x.shape}, y:{_y.shape}")

# 保存 scaler
with open(os.path.join(save_path, "scaler.pkl"), "wb") as f:
    pickle.dump(scaler, f)
hues.success(f"Saved scaler.pkl -> {save_path}")

18:58:17 - INFO - Successfully Created the folder: ['E:\DataForCode/5_AnomalyDetection/WADI_STGNN_[48_12_10]/'].
18:58:25 - SUCCESS - Saved train.npz -> x:(84613, 48, 98, 1), y:(84613, 12, 98, 1)
18:58:28 - SUCCESS - Saved val.npz -> x:(36229, 48, 98, 1), y:(36229, 12, 98, 1)
18:58:30 - SUCCESS - Saved test.npz -> x:(17222, 48, 98, 1), y:(17222, 12, 98, 1)
18:58:30 - SUCCESS - Saved scaler.pkl -> E:\DataForCode/5_AnomalyDetection/WADI_STGNN_[48_12_10]/


In [13]:
# =========================
# 10) （可选）保存一些元信息，方便你以后对齐列顺序/类型
# =========================
meta = {
    "window_size": WINDOW_SIZE,
    "horizon_size": HORIZON_SIZE,
    "downsample_sec": DOWNSAMPLE_SEC,
    "feature_cols": feature_cols,
    "status_cols": status_cols,
    "cont_cols": cont_cols,
    "train_ratio_eff": TRAIN_RATIO
}
with open(os.path.join(save_path, "meta.pkl"), "wb") as f:
    pickle.dump(meta, f)
hues.success("Saved meta.pkl (feature order & config).")

18:58:30 - SUCCESS - Saved meta.pkl (feature order & config).
